# ПЗ 7 — Распознавание объектов через Gemini

Отправляем кадры в Google Gemini Flash, получаем описание сцены в формате JSON.

In [ ]:
!pip install google-generativeai pillow -q

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)

BASE_DRIVE  = '/content/drive/MyDrive/cv-frames'
FRAMES_DIR  = f'{BASE_DRIVE}/кадры'
RESULTS_DIR = f'{BASE_DRIVE}/результаты'

os.makedirs(RESULTS_DIR, exist_ok=True)

frames = sorted(f for f in os.listdir(FRAMES_DIR) if f.endswith('.jpg'))
print(f'кадров: {len(frames)}')


In [ ]:
# читаем ключ из .env файла в Drive
# получить ключ: https://aistudio.google.com/app/apikey
env_path = '/content/drive/MyDrive/cv-frames/.env'

with open(env_path) as f:
    for line in f:
        line = line.strip()
        if line and not line.startswith('#') and '=' in line:
            key, _, val = line.partition('=')
            os.environ[key.strip()] = val.strip()

GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY', '')
print(f'ключ загружен: {GEMINI_API_KEY[:8]}...')


In [ ]:
import json
import google.generativeai as genai
from PIL import Image
from tqdm.notebook import tqdm

genai.configure(api_key=GEMINI_API_KEY)
gemini = genai.GenerativeModel('gemini-1.5-flash')

def describe_frame(image_path):
    img = Image.open(image_path)
    prompt = 'Опиши объекты на кадре. Верни JSON: {"objects": [...], "scene": "...", "mood": "..."}'
    resp = gemini.generate_content([img, prompt])
    try:
        text = resp.text.strip().lstrip('```json').rstrip('```')
        return json.loads(text)
    except Exception:
        return {'raw': resp.text}

results = []
for fname in tqdm(frames[:10], desc='gemini'):
    res = describe_frame(f'{FRAMES_DIR}/{fname}')
    res['frame'] = fname
    results.append(res)
    print(f'{fname}: {res}')

with open(f'{RESULTS_DIR}/llm_detections.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f'обработано: {len(results)} кадров')
